# Swin Transformer Explainability & Report Generation Demo

This notebook demonstrates:
1. Three explainability methods (Attention Rollout, Attention GradCAM, Multi-Head)
2. Medical report generation
3. Quantitative evaluation metrics

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

# Import our modules
from swin_explainability import UnifiedSwinVisualizer
from swin_report_generation import SwinMedicalReportGenerator
from model.swin_transformer import swin_base_patch4_window7_224

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Load Model and Image

In [ ]:
# Load Swin model
model = swin_base_patch4_window7_224(num_classes=8)
# model.load_state_dict(torch.load('path/to/checkpoint.pth'))
model = model.to(device)
model.eval()

# Load and preprocess image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load your image
# image_path = 'path/to/breakhis/image.png'
# original_image = Image.open(image_path).convert('RGB')
# image_tensor = transform(original_image).unsqueeze(0).to(device)
# original_np = np.array(original_image.resize((224, 224))) / 255.0

# For demo: use random image
image_tensor = torch.randn(1, 3, 224, 224).to(device)
original_np = np.random.rand(224, 224, 3)

## 2. Generate Comprehensive Explainability Visualization

In [ ]:
# Initialize unified visualizer
visualizer = UnifiedSwinVisualizer(model)

# Generate comprehensive visualization
fig = visualizer.generate_comprehensive_visualization(
    image_tensor,
    original_np,
    target_class=None,  # Auto-detect
    save_path='swin_explainability_demo.png',
    dpi=300
)

plt.show()

## 3. Individual Explainability Methods

In [ ]:
from swin_explainability import SwinAttentionRollout, SwinAttentionGradCAM, SwinMultiHeadVisualization

# Attention Rollout
rollout = SwinAttentionRollout(model)
rollout_map = rollout.generate_rollout(image_tensor)

# Attention GradCAM
gradcam = SwinAttentionGradCAM(model)
gradcam_map = gradcam.generate_cam(image_tensor, target_class=1)

# Multi-Head Visualization
multihead = SwinMultiHeadVisualization(model)
head_maps = multihead.visualize_heads(image_tensor, num_heads=6)

# Plot individual results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original_np)
axes[0].imshow(rollout_map, cmap='jet', alpha=0.5)
axes[0].set_title('Attention Rollout')
axes[0].axis('off')

axes[1].imshow(original_np)
axes[1].imshow(gradcam_map, cmap='jet', alpha=0.5)
axes[1].set_title('Attention GradCAM')
axes[1].axis('off')

axes[2].imshow(original_np)
axes[2].imshow(head_maps[0], cmap='viridis', alpha=0.5)
axes[2].set_title('Head 1')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 4. Analyze Head Diversity

In [ ]:
# Analyze what different heads learn
diversity = multihead.analyze_head_diversity(image_tensor)

print("Head Diversity Analysis:")
print(f"Number of heads: {diversity['num_heads']}")
print(f"Average correlation: {diversity['avg_correlation']:.3f}")
print(f"Average entropy: {diversity['avg_entropy']:.3f}")
print(f"Diversity score: {diversity['diversity_score']:.3f}")

## 5. Generate Medical Report

In [ ]:
# Initialize report generator
generator = SwinMedicalReportGenerator(model, llm_name='template')
generator = generator.to(device)

# Generate report
result = generator(image_tensor)

# Print formatted report
print(generator.format_report_text(result))

## 6. Access Structured Report Data

In [ ]:
report = result['report']

print(f"Diagnosis: {report['diagnosis']['primary']}")
print(f"Category: {report['diagnosis']['category']}")
print(f"Confidence: {report['diagnosis']['confidence']}")
print(f"\nKey Observations:")
for obs in report['key_observations']:
    print(f"  - {obs}")

print(f"\nClass Probabilities:")
class_names = generator.class_names
for i, (name, prob) in enumerate(zip(class_names, result['class_probabilities'])):
    print(f"  {name}: {prob*100:.2f}%")

## 7. Compare Explainability Methods

In [ ]:
# Compare methods with metrics
comparison = visualizer.compare_methods(image_tensor, original_np)

print("Method Comparison:")
print(f"Rollout-GradCAM Correlation: {comparison['rollout_gradcam_correlation']:.3f}")
print(f"Rollout Sparsity: {comparison['rollout_sparsity']:.3f}")
print(f"GradCAM Sparsity: {comparison['gradcam_sparsity']:.3f}")
print(f"Rollout Entropy: {comparison['rollout_entropy']:.3f}")
print(f"GradCAM Entropy: {comparison['gradcam_entropy']:.3f}")
print(f"\nHead Diversity: {comparison['head_diversity']['diversity_score']:.3f}")

## 8. Quantitative Evaluation (Optional)

In [ ]:
# This requires the evaluation script
# Uncomment to run quantitative evaluation

# from scripts.evaluate_swin_explainability import ExplainabilityEvaluator

# evaluator = ExplainabilityEvaluator(model, device)

# # Compute metrics
# insertion_auc = evaluator.insertion_auc(image_tensor, rollout_map)
# deletion_auc = evaluator.deletion_auc(image_tensor, rollout_map)
# stability = evaluator.stability_score(image_tensor, rollout)

# print(f"Insertion AUC: {insertion_auc:.3f}")
# print(f"Deletion AUC: {deletion_auc:.3f}")
# print(f"Stability: {stability:.3f}")